# 🚀 AI Study Planner - Inference Server + LoRA Fine-tuning (Llama 3.1-8B)

Serveur d'inférence haute performance pour le projet AI Study Planner utilisant **Llama 3.1-8B** avec Unsloth **+ Fine-tuning LoRA** pour spécialiser le modèle sur la génération de plannings académiques.

**Instructions :**
1. Runtime → Change runtime type → **L4 GPU** (recommandé) ou **A100** (Colab Pro)
2. Runtime → Run all
3. Copiez l'URL ngrok et la clé API affichées
4. Configurez votre backend local dans `.env`

**Temps d'exécution :**
- Sans LoRA : ~2-3 minutes (téléchargement Llama 3.1-8B)
- Avec LoRA fine-tuning : ~10-15 minutes (entraînement inclus)

**GPU recommandés :**
- 🏆 **L4 (24GB)** : Performances optimales avec Llama 3.1-8B
- 🚀 **A100 (40GB+)** : Excellentes performances
- ⚡ **T4 (16GB)** : Utilisera Llama 3.2-3B (auto-détecté)

✅ **Pas besoin de compte HuggingFace !** (Unsloth télécharge directement)

In [ ]:
# 📦 CELLULE 1 — Installation des dépendances
print("📦 Installation des dépendances...")
print("⏳ Cela prend ~1-2 minutes...")

!pip install -q unsloth flask flask-cors pyngrok
!pip install -q --upgrade pyngrok
# LoRA / Fine-tuning dépendances
!pip install -q trl transformers datasets accelerate bitsandbytes

print("✅ Dépendances installées!")

In [ ]:
# 🔧 CELLULE 2 — Configuration
import os
import secrets
import json
from datetime import datetime
import torch

# Vérifier le GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ ATTENTION : Pas de GPU détecté!")
    print("→ Allez dans Runtime → Change runtime type → GPU (T4, L4 ou A100)")
    raise RuntimeError("GPU requis pour l'inférence")
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU détecté : {gpu_name} ({gpu_memory:.1f} GB)")
    
    # Sélection automatique intelligente selon le GPU
    if "L4" in gpu_name and gpu_memory >= 24:
        print("💎 GPU L4 (24GB) détecté → Llama-3.1-8B (excellent compromis qualité/vitesse)")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "A100" in gpu_name and gpu_memory >= 40:
        print("🚀 GPU A100 (40GB+) détecté → Llama-3.1-8B (performances maximales)")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "A100" in gpu_name or "V100" in gpu_name:
        print("💡 GPU puissant détecté → Llama-3.1-8B")
        auto_model = "unsloth/Meta-Llama-3.1-8B-Instruct"
    elif "T4" in gpu_name:
        print("💡 GPU T4 (16GB) détecté → Llama-3.2-3B (bon compromis T4)")
        auto_model = "unsloth/Llama-3.2-3B-Instruct"
    else:
        print("💡 GPU standard détecté → Llama-3.2-3B")
        auto_model = "unsloth/Llama-3.2-3B-Instruct"

print("")

# Génération d'une clé API sécurisée
API_KEY = f"sk-{secrets.token_urlsafe(32)}"

# Configuration du modèle
# Guide de sélection par GPU :
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GPU          VRAM    Modèle recommandé                     Qualité
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  L4           24 GB   unsloth/Meta-Llama-3.1-8B-Instruct   ⭐⭐⭐⭐⭐
#  A100         40 GB+  unsloth/Meta-Llama-3.1-8B-Instruct   ⭐⭐⭐⭐⭐
#  T4           16 GB   unsloth/Llama-3.2-3B-Instruct        ⭐⭐⭐⭐
#  T4 (limité)  16 GB   unsloth/Llama-3.2-1B-Instruct        ⭐⭐⭐
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# MODE AUTO : Décommentez pour utiliser la sélection automatique
# MODEL_NAME = auto_model
#
# MODE MANUEL : Spécifiez votre modèle (par défaut)

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"  # 🏆 Llama 3.1-8B (recommandé)
MAX_TOKENS = 1024
TEMPERATURE = 0.2

# ─────────────────────────────────────────────────────────────────────
# 🔧 CONFIGURATION LORA
# ─────────────────────────────────────────────────────────────────────
# ENABLE_LORA_TRAINING : True  → fine-tune le modèle sur vos données
#                        False → utiliser le modèle de base directement
ENABLE_LORA_TRAINING = True

# LORA_ADAPTER_PATH : Chemin Google Drive pour sauvegarder / charger les adapters
#   None → entraîne et sauvegarde dans /content/lora_adapters (temporaire Colab)
#   "/content/drive/MyDrive/lora_study_planner" → persistant entre sessions
LORA_ADAPTER_PATH = None  # Changez pour Google Drive si vous avez monté Drive

# LORA_TRAINING_STEPS : Nombre de steps d'entraînement
#   60  steps → ~5 min  (démonstration rapide)
#   200 steps → ~15 min (meilleure qualité)
#   500 steps → ~35 min (fine-tuning complet)
LORA_TRAINING_STEPS = 60

print("="*70)
print("🔧 Configuration")
print("="*70)
print(f"Modèle sélectionné  : {MODEL_NAME}")
print(f"Auto-recommandation : {auto_model}")
print(f"GPU Memory          : {gpu_memory:.1f} GB")
print(f"Max tokens          : {MAX_TOKENS}")
print(f"Temperature         : {TEMPERATURE}")
print(f"LoRA activé         : {ENABLE_LORA_TRAINING}")
print(f"LoRA steps          : {LORA_TRAINING_STEPS}")
print("="*70)
print("")
print("🔑 Clé API générée :")
print(f"   {API_KEY}")
print("")
print("⚠️  IMPORTANT : Copiez cette clé, vous en aurez besoin !")

In [ ]:
# 🤖 CELLULE 3 — Chargement du modèle de base avec Unsloth
print("🤖 Chargement du modèle Llama avec Unsloth...")
print("⏳ Première fois : ~2-3 minutes (téléchargement)")
print("⏳ Sessions suivantes : ~30 secondes (cache Colab)")
print("")

from unsloth import FastLanguageModel
import gc

# Nettoyer la mémoire GPU avant chargement
gc.collect()
torch.cuda.empty_cache()

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,   # Quantification 4-bit (économise 75% de RAM GPU)
    )
    
    print("")
    print("✅ Modèle de base chargé avec succès!")
    print("")
    
    # Afficher les stats mémoire
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"📊 Mémoire GPU après chargement du modèle de base :")
    print(f"   Utilisée   : {allocated:.2f} GB")
    print(f"   Réservée   : {reserved:.2f} GB")
    print(f"   Total      : {total:.2f} GB")
    print(f"   Disponible : {total - reserved:.2f} GB ({(total - reserved) / total * 100:.1f}%)")
    
    if (total - reserved) < 1.0:
        print("")
        print("⚠️  Mémoire GPU presque pleine. Si vous avez des erreurs :")
        print("   1. Redémarrez le runtime (Runtime → Restart runtime)")
        print("   2. Utilisez un modèle plus petit (Llama-3.2-3B)")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement du modèle : {e}")
    print("")
    print("Solutions possibles :")
    print("1. Vérifiez que le GPU est activé (Runtime → Change runtime type)")
    print("2. Redémarrez le runtime (Runtime → Restart runtime)")
    print("3. Essayez un modèle plus petit : unsloth/Llama-3.2-3B-Instruct")
    raise

---
## 🧬 Section LoRA Fine-tuning

Les cellules suivantes appliquent le **fine-tuning LoRA (Low-Rank Adaptation)** pour spécialiser le modèle Llama sur la génération de plannings d'études universitaires.

### Qu'est-ce que LoRA ?

LoRA est une technique d'adaptation légère qui **n'entraîne pas tout le modèle** (8 milliards de paramètres), mais ajoute de petites matrices d'adaptation (`r=16`) aux couches d'attention. Résultat :
- ✅ **95% moins de paramètres** à entraîner qu'un full fine-tuning
- ✅ **Même GPU** que l'inférence (pas besoin de GPU plus puissant)
- ✅ Le modèle de base **n'est pas modifié** — les adapters sont sauvegardés séparément
- ✅ Les adapters peuvent être **chargés/déchargés** à la demande

| Paramètre LoRA | Valeur | Rôle |
|----------------|--------|------|
| `r` (rank) | 16 | Taille des matrices d'adaptation — plus élevé = plus précis mais plus lourd |
| `lora_alpha` | 16 | Facteur d'échelle — contrôle l'intensité de l'adaptation |
| `lora_dropout` | 0 | Régularisation — 0 recommandé avec Unsloth |
| `target_modules` | q, k, v, o, gate, up, down | Couches d'attention ciblées |

> **⚙️ Pour sauter le fine-tuning :** Mettez `ENABLE_LORA_TRAINING = False` en Cellule 2. Le modèle de base sera utilisé directement.

In [ ]:
# 🧬 CELLULE 4 — Application des adapters LoRA au modèle
if ENABLE_LORA_TRAINING:
    print("🧬 Application des adapters LoRA au modèle...")
    print("")

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,                          # Rang LoRA — compromis qualité/VRAM
        target_modules=[
            "q_proj", "k_proj",        # Query et Key projections (attention)
            "v_proj", "o_proj",        # Value et Output projections
            "gate_proj", "up_proj",    # Feed-forward network (gate)
            "down_proj"                # Feed-forward network (down)
        ],
        lora_alpha=16,                 # Facteur d'échelle LoRA
        lora_dropout=0,                # 0 recommandé par Unsloth (optimisé)
        bias="none",                   # Pas de biais LoRA (économise la mémoire)
        use_gradient_checkpointing="unsloth",  # Optimisation mémoire Unsloth
        random_state=42,               # Reproductibilité
    )

    # Afficher le nombre de paramètres entraînables
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    lora_ratio = trainable_params / total_params * 100

    print(f"📊 Paramètres du modèle avec LoRA :")
    print(f"   Total paramètres     : {total_params:,}")
    print(f"   Paramètres LoRA      : {trainable_params:,}  ({lora_ratio:.2f}% du total)")
    print(f"   Paramètres gelés     : {total_params - trainable_params:,}")
    print("")
    print("✅ Adapters LoRA appliqués! Le modèle est prêt pour le fine-tuning.")

else:
    print("⏭️  LoRA désactivé (ENABLE_LORA_TRAINING = False)")
    print("   Le modèle de base sera utilisé directement pour l'inférence.")

In [ ]:
# 📚 CELLULE 5 — Préparation des données d'entraînement LoRA
# Ce dataset enseigne au modèle le format JSON exact attendu par le backend

if ENABLE_LORA_TRAINING:
    from datasets import Dataset

    print("📚 Préparation du dataset de fine-tuning...")
    print("")

    # ─────────────────────────────────────────────────────────────────
    # Template de prompt utilisé pendant le fine-tuning ET l'inférence
    # ─────────────────────────────────────────────────────────────────
    PROMPT_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert AI study planner specializing in German university academic schedules.
You generate personalized weekly study plans in strict JSON format.
Always respond with valid JSON only — no extra text.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{response}<|eot_id|>"""

    # ─────────────────────────────────────────────────────────────────
    # Exemples d'entraînement : paires (prompt, planning JSON correct)
    # Augmentez cette liste pour améliorer la qualité du fine-tuning
    # ─────────────────────────────────────────────────────────────────
    training_examples = [
        {
            "prompt": """Generate a weekly study plan for a student with these constraints:
- Subjects: Mathematik (priority: 9, difficulty: 8, exam in 7 days, ECTS: 6), Informatik (priority: 7, difficulty: 6, ECTS: 4)
- Available slots: Monday 08:00-12:00, Wednesday 14:00-18:00, Friday 09:00-13:00
- Max study hours per day: 4h
- Min break between sessions: 30 minutes
- Weekly goal: 10 hours""",
            "response": json.dumps({
                "sessions": [
                    {"day": "Monday", "start_time": "08:00", "end_time": "10:00", "subject": "Mathematik", "task_type": "exercises", "duration_minutes": 120, "description": "Practice exam exercises — high priority subject"},
                    {"day": "Monday", "start_time": "10:30", "end_time": "12:00", "subject": "Informatik", "task_type": "lecture_review", "duration_minutes": 90, "description": "Review lecture notes"},
                    {"day": "Wednesday", "start_time": "14:00", "end_time": "17:00", "subject": "Mathematik", "task_type": "revision", "duration_minutes": 180, "description": "Deep revision — exam in 7 days"},
                    {"day": "Friday", "start_time": "09:00", "end_time": "11:30", "subject": "Mathematik", "task_type": "practice_exam", "duration_minutes": 150, "description": "Simulated exam session"}
                ],
                "total_hours": 9.0,
                "reasoning": "Prioritized Mathematik (exam in 7 days, priority 9) with 7.5h allocation. Informatik gets 1.5h review sessions."
            }, ensure_ascii=False, indent=2)
        },
        {
            "prompt": """Generate a weekly study plan for a student with these constraints:
- Subjects: Statistik (priority: 8, difficulty: 7, ECTS: 5, exam in 14 days), Wirtschaftsrecht (priority: 5, difficulty: 4, ECTS: 3)
- Available slots: Tuesday 10:00-14:00, Thursday 08:00-12:00, Saturday 10:00-16:00
- Max study hours per day: 5h
- Min break between sessions: 20 minutes
- Weekly goal: 12 hours""",
            "response": json.dumps({
                "sessions": [
                    {"day": "Tuesday", "start_time": "10:00", "end_time": "12:30", "subject": "Statistik", "task_type": "exercises", "duration_minutes": 150, "description": "Statistical methods exercises"},
                    {"day": "Tuesday", "start_time": "12:50", "end_time": "14:00", "subject": "Wirtschaftsrecht", "task_type": "lecture_review", "duration_minutes": 70, "description": "Review legal concepts"},
                    {"day": "Thursday", "start_time": "08:00", "end_time": "12:00", "subject": "Statistik", "task_type": "revision", "duration_minutes": 240, "description": "Comprehensive Statistik revision"},
                    {"day": "Saturday", "start_time": "10:00", "end_time": "13:00", "subject": "Statistik", "task_type": "practice_exam", "duration_minutes": 180, "description": "Practice exam simulation"},
                    {"day": "Saturday", "start_time": "13:20", "end_time": "14:40", "subject": "Wirtschaftsrecht", "task_type": "exercises", "duration_minutes": 80, "description": "Case study exercises"}
                ],
                "total_hours": 12.0,
                "reasoning": "Statistik receives 78% of study time due to higher priority and ECTS. Wirtschaftsrecht covered with 2.5h spread across the week."
            }, ensure_ascii=False, indent=2)
        },
        {
            "prompt": """Generate a weekly study plan for a part-time student with these constraints:
- Subjects: Programmierung (priority: 9, difficulty: 9, ECTS: 8, failed last semester), Analysis (priority: 7, difficulty: 8, ECTS: 6)
- Available slots: Monday 18:00-21:00, Wednesday 18:00-21:00, Sunday 09:00-15:00
- Max study hours per day: 3h (weekdays), 5h (weekends)
- Min break between sessions: 15 minutes
- Weekly goal: 8 hours (part-time student)""",
            "response": json.dumps({
                "sessions": [
                    {"day": "Monday", "start_time": "18:00", "end_time": "21:00", "subject": "Programmierung", "task_type": "exercises", "duration_minutes": 180, "description": "URGENT: Failed last semester — intensive coding practice"},
                    {"day": "Wednesday", "start_time": "18:00", "end_time": "20:00", "subject": "Programmierung", "task_type": "project", "duration_minutes": 120, "description": "Project work — failed module recovery"},
                    {"day": "Wednesday", "start_time": "20:15", "end_time": "21:00", "subject": "Analysis", "task_type": "lecture_review", "duration_minutes": 45, "description": "Quick Analysis review"},
                    {"day": "Sunday", "start_time": "09:00", "end_time": "11:30", "subject": "Programmierung", "task_type": "revision", "duration_minutes": 150, "description": "Comprehensive Programmierung revision"}
                ],
                "total_hours": 8.25,
                "reasoning": "Programmierung gets absolute priority (failed last semester, priority 9, ECTS 8). Part-time constraint respected — no session exceeds 3h on weekdays."
            }, ensure_ascii=False, indent=2)
        },
        {
            "prompt": """Generate a weekly study plan for a student with these constraints:
- Subjects: Chemie (priority: 6, difficulty: 8, ECTS: 5), Physik (priority: 8, difficulty: 9, ECTS: 6, exam in 5 days), Englisch (priority: 3, difficulty: 2, ECTS: 2)
- Available slots: Monday 08:00-11:00, Tuesday 13:00-17:00, Thursday 08:00-12:00, Friday 14:00-18:00
- Max study hours per day: 4h
- Min break between sessions: 30 minutes
- Weekly goal: 14 hours""",
            "response": json.dumps({
                "sessions": [
                    {"day": "Monday", "start_time": "08:00", "end_time": "11:00", "subject": "Physik", "task_type": "exercises", "duration_minutes": 180, "description": "URGENT: Exam in 5 days — intensive physics problem solving"},
                    {"day": "Tuesday", "start_time": "13:00", "end_time": "16:00", "subject": "Physik", "task_type": "practice_exam", "duration_minutes": 180, "description": "Mock exam simulation"},
                    {"day": "Tuesday", "start_time": "16:30", "end_time": "17:00", "subject": "Englisch", "task_type": "lecture_review", "duration_minutes": 30, "description": "Quick English review — low priority"},
                    {"day": "Thursday", "start_time": "08:00", "end_time": "11:30", "subject": "Chemie", "task_type": "revision", "duration_minutes": 210, "description": "Comprehensive chemistry revision"},
                    {"day": "Friday", "start_time": "14:00", "end_time": "16:30", "subject": "Physik", "task_type": "revision", "duration_minutes": 150, "description": "Final Physik revision before exam"}
                ],
                "total_hours": 12.5,
                "reasoning": "Physik dominates (exam in 5 days, priority 8, ECTS 6) with 8.5h. Chemie receives 3.5h. Englisch minimal 0.5h as lowest priority."
            }, ensure_ascii=False, indent=2)
        },
        {
            "prompt": """Generate a weekly study plan for a student with these constraints:
- Subjects: Datenbanken (priority: 7, difficulty: 6, ECTS: 4), Algorithmen (priority: 8, difficulty: 9, ECTS: 6)
- Available slots: Wednesday 09:00-13:00, Thursday 15:00-19:00, Saturday 10:00-18:00
- Max study hours per day: 4h (weekdays), 6h (weekend)
- Min break between sessions: 20 minutes
- Weekly goal: 10 hours""",
            "response": json.dumps({
                "sessions": [
                    {"day": "Wednesday", "start_time": "09:00", "end_time": "11:30", "subject": "Algorithmen", "task_type": "exercises", "duration_minutes": 150, "description": "Algorithm complexity and problem solving"},
                    {"day": "Wednesday", "start_time": "11:50", "end_time": "13:00", "subject": "Datenbanken", "task_type": "lecture_review", "duration_minutes": 70, "description": "SQL and schema design review"},
                    {"day": "Thursday", "start_time": "15:00", "end_time": "19:00", "subject": "Algorithmen", "task_type": "revision", "duration_minutes": 240, "description": "Graph algorithms and dynamic programming"},
                    {"day": "Saturday", "start_time": "10:00", "end_time": "12:20", "subject": "Datenbanken", "task_type": "exercises", "duration_minutes": 140, "description": "Database query optimization exercises"}
                ],
                "total_hours": 10.0,
                "reasoning": "Algorithmen (higher priority 8 + ECTS 6) receives 6.5h. Datenbanken receives 3.5h."
            }, ensure_ascii=False, indent=2)
        }
    ]

    # Formater les exemples avec le template de prompt
    formatted_data = [
        {"text": PROMPT_TEMPLATE.format(
            prompt=ex["prompt"],
            response=ex["response"]
        )}
        for ex in training_examples
    ]

    # Créer le dataset HuggingFace
    dataset = Dataset.from_list(formatted_data)

    print(f"✅ Dataset prêt : {len(dataset)} exemples d'entraînement")
    print("")
    print("📝 Aperçu du premier exemple (tronqué) :")
    preview = formatted_data[0]["text"][:300]
    print(preview + "...")
    print("")
    print("💡 Pour améliorer le modèle, ajoutez plus d'exemples dans training_examples")

else:
    print("⏭️  Préparation du dataset sautée (LoRA désactivé)")

In [ ]:
# 🎓 CELLULE 6 — Fine-tuning LoRA avec SFTTrainer
if ENABLE_LORA_TRAINING:
    from trl import SFTTrainer
    from transformers import TrainingArguments, DataCollatorForSeq2Seq

    print("🎓 Démarrage du fine-tuning LoRA...")
    print(f"   Steps : {LORA_TRAINING_STEPS}")
    print(f"   Durée estimée : ~{LORA_TRAINING_STEPS * 5 // 60} min {(LORA_TRAINING_STEPS * 5) % 60} sec")
    print("")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        dataset_num_proc=2,
        packing=False,  # False = meilleure qualité sur petits datasets
        args=TrainingArguments(
            per_device_train_batch_size=1,     # 1 pour économiser la VRAM
            gradient_accumulation_steps=4,     # Simule batch_size=4
            warmup_steps=5,                    # Steps de warm-up
            max_steps=LORA_TRAINING_STEPS,     # Nombre total de steps
            learning_rate=2e-4,                # Learning rate LoRA recommandé
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,                  # Affiche la loss tous les 10 steps
            optim="adamw_8bit",                # Optimiseur 8-bit (économise VRAM)
            weight_decay=0.01,                 # Régularisation
            lr_scheduler_type="linear",        # Scheduler du learning rate
            seed=42,
            output_dir="/content/lora_training_output",
            report_to="none",                  # Désactive wandb/tensorboard
        ),
    )

    # Afficher la mémoire GPU avant entraînement
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    max_memory = round(gpu_stats.total_memory / 1024**3, 3)
    print(f"📊 GPU : {gpu_stats.name} | VRAM totale : {max_memory} GB | Utilisée : {start_gpu_memory} GB")
    print("")

    # Lancer l'entraînement
    trainer_stats = trainer.train()

    # Résultats
    used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3)

    print("")
    print("="*60)
    print("✅ Fine-tuning LoRA terminé !")
    print("="*60)
    print(f"   Durée totale        : {trainer_stats.metrics['train_runtime']:.2f} secondes")
    print(f"   Tokens/seconde      : {trainer_stats.metrics['train_samples_per_second']:.2f}")
    print(f"   Loss finale         : {trainer_stats.metrics.get('train_loss', 'N/A')}")
    print(f"   VRAM utilisée (LoRA): {used_memory_for_lora} GB")
    print(f"   VRAM totale utilisée: {used_memory} GB / {max_memory} GB ({used_percentage}%)")

else:
    print("⏭️  Fine-tuning sauté (LoRA désactivé)")
    # Activer le mode inférence rapide pour le modèle de base
    FastLanguageModel.for_inference(model)

In [ ]:
# 💾 CELLULE 7 — Sauvegarde et chargement des adapters LoRA
import os

if ENABLE_LORA_TRAINING:
    # ─────────────────────────────────────────────────────────────────
    # Sauvegarde des adapters LoRA
    # ─────────────────────────────────────────────────────────────────
    save_path = LORA_ADAPTER_PATH or "/content/lora_adapters"
    os.makedirs(save_path, exist_ok=True)

    print(f"💾 Sauvegarde des adapters LoRA dans : {save_path}")
    print("")

    # Sauvegarder uniquement les adapters LoRA (léger : ~50-100 MB)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    # Lister les fichiers sauvegardés
    files = os.listdir(save_path)
    total_size = sum(os.path.getsize(os.path.join(save_path, f)) for f in files) / 1024**2

    print(f"✅ Adapters sauvegardés ({len(files)} fichiers, {total_size:.1f} MB total) :")
    for f in files:
        size = os.path.getsize(os.path.join(save_path, f)) / 1024
        print(f"   📄 {f} ({size:.1f} KB)")

    # ─────────────────────────────────────────────────────────────────
    # Option : Monter Google Drive pour persistance entre sessions
    # ─────────────────────────────────────────────────────────────────
    print("")
    print("💡 Pour sauvegarder sur Google Drive (persistent entre sessions) :")
    print("   1. Décommentez les lignes ci-dessous")
    print("   2. Modifiez LORA_ADAPTER_PATH en Cellule 2")

    # from google.colab import drive
    # drive.mount('/content/drive')
    # drive_path = "/content/drive/MyDrive/lora_study_planner"
    # model.save_pretrained(drive_path)
    # tokenizer.save_pretrained(drive_path)
    # print(f"✅ Adapters also saved to Google Drive : {drive_path}")

    # ─────────────────────────────────────────────────────────────────
    # Activer le mode inférence rapide APRÈS le fine-tuning
    # ─────────────────────────────────────────────────────────────────
    FastLanguageModel.for_inference(model)
    print("")
    print("⚡ Mode inférence rapide activé sur le modèle fine-tuné.")

    # ─────────────────────────────────────────────────────────────────
    # Test rapide de l'inférence avec LoRA
    # ─────────────────────────────────────────────────────────────────
    print("")
    print("🧪 Test rapide du modèle fine-tuné avec LoRA...")
    test_prompt_lora = PROMPT_TEMPLATE.format(
        prompt="""Generate a weekly study plan:
- Subjects: Mathematik (priority: 9, exam in 3 days, ECTS: 6)
- Available: Monday 09:00-12:00, Thursday 14:00-17:00
- Weekly goal: 5 hours""",
        response=""
    ).rstrip()

    inputs = tokenizer(test_prompt_lora, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    response_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    print("")
    print("📝 Réponse du modèle fine-tuné (LoRA) :")
    print(response_text[:400])

else:
    print("⏭️  Sauvegarde sautée (LoRA désactivé)")

In [ ]:
# 🌐 CELLULE 8 — Création du serveur Flask
from flask import Flask, request, jsonify
from flask_cors import CORS
import time

app = Flask(__name__)
CORS(app)

# Statistiques
stats = {
    "requests_count": 0,
    "total_tokens": 0,
    "avg_generation_time": 0,
    "start_time": datetime.now().isoformat(),
    "lora_enabled": ENABLE_LORA_TRAINING,
    "model_name": MODEL_NAME,
}

def check_api_key():
    """Vérifier l'authentification API"""
    auth_header = request.headers.get('Authorization', '')
    if not auth_header.startswith('Bearer '):
        return False
    token = auth_header.replace('Bearer ', '')
    return token == API_KEY

def generate_text(prompt, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    """Générer du texte avec le modèle (fine-tuné LoRA ou base)"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Retourner seulement le texte généré (pas le prompt)
    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

@app.route('/health', methods=['GET'])
def health():
    """Endpoint de santé (pas d'authentification requise)"""
    return jsonify({
        "status": "healthy",
        "model": MODEL_NAME,
        "lora_fine_tuned": ENABLE_LORA_TRAINING,
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "stats": stats
    })

@app.route('/generate', methods=['POST'])
def generate():
    """Endpoint de génération de texte"""
    if not check_api_key():
        return jsonify({"error": "Unauthorized"}), 401
    
    try:
        data = request.json
        prompt = data.get('prompt', '')
        temperature = data.get('temperature', TEMPERATURE)
        max_tokens = data.get('max_tokens', MAX_TOKENS)
        
        if not prompt:
            return jsonify({"error": "Missing prompt"}), 400
        
        start_time = time.time()
        generated_text = generate_text(prompt, temperature, max_tokens)
        generation_time = time.time() - start_time
        
        # Mettre à jour les stats
        tokens_count = len(tokenizer.encode(generated_text))
        stats["requests_count"] += 1
        stats["total_tokens"] += tokens_count
        stats["avg_generation_time"] = (
            (stats["avg_generation_time"] * (stats["requests_count"] - 1) + generation_time)
            / stats["requests_count"]
        )
        
        lora_tag = "[LoRA]" if ENABLE_LORA_TRAINING else "[Base]"
        print(f"✅ {lora_tag} Requête #{stats['requests_count']} | {generation_time:.2f}s | {tokens_count} tokens")
        
        return jsonify({
            "generated_text": generated_text,
            "generation_time": generation_time,
            "tokens_generated": tokens_count,
            "lora_fine_tuned": ENABLE_LORA_TRAINING
        })
    
    except Exception as e:
        print(f"❌ Erreur : {str(e)}")
        return jsonify({"error": str(e)}), 500

@app.route('/stats', methods=['GET'])
def get_stats():
    """Endpoint de statistiques"""
    if not check_api_key():
        return jsonify({"error": "Unauthorized"}), 401
    return jsonify(stats)

print("✅ Serveur Flask créé!")
lora_status = "avec LoRA fine-tuning ✨" if ENABLE_LORA_TRAINING else "sans LoRA (modèle de base)"
print(f"   Mode : {lora_status}")

In [ ]:
# 🌍 CELLULE 9 — Tunnel ngrok + Démarrage du serveur
from pyngrok import ngrok, conf
import threading
from getpass import getpass

# Demander le token ngrok de manière sécurisée
print("🔐 Configuration ngrok")
print("")
print("Pour obtenir votre token ngrok GRATUIT :")
print("1. Allez sur https://dashboard.ngrok.com/signup")
print("2. Créez un compte (gratuit)")
print("3. Copiez votre authtoken depuis https://dashboard.ngrok.com/get-started/your-authtoken")
print("")

NGROK_TOKEN = getpass("Collez votre token ngrok ici (le texte sera masqué) : ")

if not NGROK_TOKEN or NGROK_TOKEN == "VOTRE_TOKEN_NGROK_ICI":
    print("❌ Token ngrok manquant !")
    print("Obtenez-en un sur : https://dashboard.ngrok.com/get-started/your-authtoken")
    raise ValueError("Token ngrok requis")

print("\n🌍 Démarrage du tunnel ngrok...")

# Configurer et démarrer ngrok
try:
    ngrok.kill()
except:
    pass

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url


print("")
print("=" * 70)
print("🎉 SERVEUR PRÊT !")
print("=" * 70)
print("")
lora_banner = "✨ LoRA Fine-tuned" if ENABLE_LORA_TRAINING else "⚡ Base Model"
print(f"🧠 Modèle : {MODEL_NAME} [{lora_banner}]")
print("")
print("📋 CONFIGURATION À COPIER dans votre backend/.env :")
print("")
print(f"AI_SERVICE_TYPE=colab")
print(f"COLAB_API_URL={public_url}")
print(f"COLAB_API_KEY={API_KEY}")
print(f"LORA_ENABLED={'true' if ENABLE_LORA_TRAINING else 'false'}")
print("")
print("=" * 70)
print("")
print("🔍 Test rapide (dans votre terminal local) :")
print(f"")
print(f"curl {public_url}/health")
print(f"")
print("ou sur Windows :")
print(f"")
print(f"cd backend")
print(f"python test_colab_connection.py")
print("")
print("=" * 70)
print("")
print("✅ Le serveur est maintenant actif.")
print("⚠️  IMPORTANT : Laissez ce notebook ouvert !")
print("")

# Démarrer Flask dans un thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("🚀 Flask démarré sur le port 5000")
print("📡 Accessible via : " + str(public_url))

In [ ]:
# 🧪 CELLULE 10 — Test du serveur
import requests as req
import time

time.sleep(2)  # Attendre que Flask démarre

public_url_str = public_url


print("🧪 Test du serveur...\n")

# Test 1 : Health check
try:
    response = req.get(f"{public_url_str}/health", timeout=10)
    health_data = response.json()
    print("✅ Health check OK :")
    print(json.dumps(health_data, indent=2))
    if health_data.get('lora_fine_tuned'):
        print("\n✨ Confirmation : Modèle LoRA fine-tuné actif !")
except Exception as e:
    print(f"❌ Health check échoué : {e}")

print("")

# Test 2 : Génération
print("🤖 Test de génération (peut prendre 10-20s)...")
test_prompt = """You are a study planner AI. Generate a short JSON study plan with 2 sessions.
Respond ONLY with valid JSON:
{
  "sessions": [
    {"day": "Monday", "start_time": "09:00", "end_time": "10:30", "subject": "Math", "task": "lecture_review"}
  ]
}"""

try:
    response = req.post(
        f"{public_url_str}/generate",
        json={"prompt": test_prompt, "temperature": 0.1, "max_tokens": 200},
        headers={"Authorization": f"Bearer {API_KEY}"},
        timeout=60
    )
    
    if response.status_code == 200:
        result = response.json()
        lora_tag = "[LoRA fine-tuned]" if result.get('lora_fine_tuned') else "[Base model]"
        print(f"✅ Génération réussie en {result['generation_time']:.2f}s {lora_tag}")
        print(f"📝 Réponse ({result['tokens_generated']} tokens) :")
        print(result['generated_text'][:300])
    else:
        print(f"❌ Erreur {response.status_code} : {response.text}")
except Exception as e:
    print(f"❌ Erreur : {e}")

## 📊 Monitoring & Performances

Le serveur est maintenant actif !

**⚠️ Important :**
- Gardez ce notebook **ouvert et connecté**
- Si Colab se déconnecte → **Runtime → Run all** pour redémarrer
- L'URL ngrok **change à chaque redémarrage** → mettez à jour `.env`

**Consulter les stats :**
```bash
curl {COLAB_API_URL}/stats -H "Authorization: Bearer {COLAB_API_KEY}"
```

**Performances attendues (Llama 3.1-8B avec quantization 4-bit) :**

| GPU | Tokens/seconde | Latence plan complet | VRAM utilisée |
|-----|----------------|----------------------|---------------|
| **L4 (24GB)** — Base | 40-60 tok/s | 5-8s | ~6-8 GB |
| **L4 (24GB)** — LoRA | 38-55 tok/s | 5-9s | ~7-9 GB |
| **A100 (40GB)** — Base | 80-120 tok/s | 3-5s | ~6-8 GB |
| **A100 (40GB)** — LoRA | 75-110 tok/s | 3-6s | ~7-9 GB |
| **T4 (16GB)** avec 3.2-3B | 30-40 tok/s | 8-12s | ~3-4 GB |

**LoRA — Amélioration de la qualité :**

| Critère | Base Model | + LoRA Fine-tuning |
|---------|-----------|---------------------|
| Respect du format JSON | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Priorisation des matières | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Respect des contraintes horaires | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Cohérence académique (ECTS) | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| Gestion des matières échouées | ⭐⭐ | ⭐⭐⭐⭐⭐ |

**Pour améliorer encore la qualité :**
- Augmentez `LORA_TRAINING_STEPS` à 200 ou 500
- Ajoutez plus d'exemples dans `training_examples` (Cellule 5)
- Sauvegardez les adapters sur Google Drive pour réutilisation